In [2]:
print("\nStarting augmentation...")
norm_files = sorted(glob.glob(os.path.join(normalized_folder, "norm_*.fits")))
total_aug = 0

for i, fpath in enumerate(norm_files, 1):
    fname = os.path.basename(fpath)
    base = os.path.splitext(fname)[0].replace('norm_', '')
    
    with fits.open(fpath) as hdul:
        data = hdul[0].data
        hdr = hdul[0].header
        
        augs = augment_image(data)
        
        for suffix, aug_data in augs:
            aug_path = os.path.join(augmented_folder, f"aug_{base}_{suffix}.fits")
            fits.PrimaryHDU(aug_data.astype(np.float32), hdr).writeto(aug_path, overwrite=True)
            total_aug += 1
        
        # Save visualization every 10th file
        if i % 10 == 0:
            fig, axes = plt.subplots(3, 3, figsize=(10, 10))
            axes = axes.flatten()
            
            axes[0].imshow(data, cmap='gray', origin='lower', vmin=0, vmax=1)
            axes[0].set_title(f'{i}. Original')
            axes[0].axis('off')
            
            for j, (suffix, aug_data) in enumerate(augs, 1):
                axes[j].imshow(aug_data, cmap='gray', origin='lower', vmin=0, vmax=1)
                axes[j].set_title(suffix)
                axes[j].axis('off')
            
            plt.tight_layout()
            plt.savefig(os.path.join(augmented_folder, f"sample_{i:03d}.png"), dpi=150)
            plt.close()
        
        if i % 10 == 0 or i == len(norm_files):
            print(f"  {i}/{len(norm_files)} done")

print(f"\nFinished!")
print(f"Normalized: {len(norm_files)} files")
print(f"Augmented: {total_aug} files")


Starting augmentation...
  10/467 done
  20/467 done
  30/467 done
  40/467 done
  50/467 done
  60/467 done
  70/467 done
  80/467 done
  90/467 done
  100/467 done
  110/467 done
  120/467 done
  130/467 done
  140/467 done
  150/467 done
  160/467 done
  170/467 done
  180/467 done
  190/467 done
  200/467 done
  210/467 done
  220/467 done
  230/467 done
  240/467 done
  250/467 done
  260/467 done
  270/467 done
  280/467 done
  290/467 done
  300/467 done
  310/467 done
  320/467 done
  330/467 done
  340/467 done
  350/467 done
  360/467 done
  370/467 done
  380/467 done
  390/467 done
  400/467 done
  410/467 done
  420/467 done
  430/467 done
  440/467 done
  450/467 done
  460/467 done
  467/467 done

Finished!
Normalized: 467 files
Augmented: 3736 files


In [ ]:
#not implemented
import os
import glob
import numpy as np
from astropy.io import fits
from scipy import ndimage
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

input_folder = r"recreate"
output_folder = r"normalised"

normalize_method = "minmax"
create_augmented = True
save_visualizations = False  # set True only for first few

os.makedirs(output_folder, exist_ok=True)

def normalize_data(data, method="minmax"):
    data = np.nan_to_num(data, 0)
    
    if method == "minmax":
        vmin, vmax = np.percentile(data, [1, 99])
        norm = (data - vmin) / (vmax - vmin + 1e-10)
        return np.clip(norm, 0, 1)
    
    elif method == "zscore":
        mean, std = np.mean(data), np.std(data)
        norm = (data - mean) / (std + 1e-10)
        norm = np.clip(norm, -3, 3)
        return (norm + 3) / 6
    
    elif method == "percentile":
        p2, p98 = np.percentile(data, [2, 98])
        norm = (data - p2) / (p98 - p2 + 1e-10)
        return np.clip(norm, 0, 1)

def augment_data(data):
    augmented = []
    for angle in [0, 90, 180, 270]:
        rotated = ndimage.rotate(data, angle, reshape=False, order=1)
        augmented.append((f'rot{angle}', rotated))
        augmented.append((f'rot{angle}_flip', np.fliplr(rotated)))
    return augmented

fits_files = glob.glob(os.path.join(input_folder, "*.fits"))
total = len(fits_files)
print(f"Found {total} FITS files")
print(f"Normalization: {normalize_method}")
print(f"Augmentation: {create_augmented}")
print("="*60)

processed = 0
augmented_count = 0
failed = []

for i, fpath in enumerate(fits_files, 1):
    fname = os.path.basename(fpath)
    base = os.path.splitext(fname)[0]
    
    # progress update every 10%
    if i % max(1, total // 10) == 0:
        print(f"Progress: {i}/{total} ({100*i//total}%)")
    
    try:
        with fits.open(fpath) as hdul:
            data = hdul[0].data
            hdr = hdul[0].header
            
            # normalize
            normalized = normalize_data(data, normalize_method)
            
            # header
            hdr_new = hdr.copy()
            hdr_new['PREPROC'] = True
            hdr_new['NORMTYPE'] = normalize_method
            
            # save original preprocessed
            out_path = os.path.join(output_folder, f"prep_{base}.fits")
            fits.PrimaryHDU(normalized.astype(np.float32), hdr_new).writeto(
                out_path, overwrite=True)
            processed += 1
            
            # augmentation
            if create_augmented:
                for suffix, aug in augment_data(normalized):
                    aug_path = os.path.join(output_folder, f"aug_{base}_{suffix}.fits")
                    fits.PrimaryHDU(aug.astype(np.float32), hdr_new).writeto(
                        aug_path, overwrite=True)
                    augmented_count += 1
            
            # save visualization only for first 3
            if save_visualizations and i <= 3:
                fig, axes = plt.subplots(1, 2, figsize=(10, 5))
                
                v1, v2 = np.percentile(data, [1, 99])
                axes[0].imshow(data, cmap='gray', vmin=v1, vmax=v2, origin='lower')
                axes[0].set_title('Original')
                axes[0].axis('off')
                
                axes[1].imshow(normalized, cmap='gray', origin='lower')
                axes[1].set_title(f'Normalized')
                axes[1].axis('off')
                
                plt.tight_layout()
                plt.savefig(os.path.join(output_folder, f"demo_{base}.png"), 
                           dpi=100, bbox_inches='tight')
                plt.close()
                
    except Exception as e:
        print(f"  ERROR: {fname} - {str(e)}")
        failed.append(fname)
        continue

print("\n" + "="*60)
print("Processing Complete!")
print(f"Successfully processed: {processed}/{total}")
if create_augmented:
    print(f"Augmented versions: {augmented_count}")
    print(f"Total files created: {processed + augmented_count}")
if failed:
    print(f"\nFailed files ({len(failed)}):")
    for f in failed:
        print(f"  - {f}")
print(f"\nOutput folder: {output_folder}")

In [ ]:
import os
import glob
import numpy as np
from astropy.io import fits
from scipy import ndimage
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

input_folder = "train_resized"
normalized_folder = "train_normalized"
augmented_folder = "train_augmented"

os.makedirs(normalized_folder, exist_ok=True)
os.makedirs(augmented_folder, exist_ok=True)

def normalize_data(data, method="minmax"):
    data = np.nan_to_num(data, 0)
    
    if method == "minmax":
        vmin, vmax = np.percentile(data, [1, 99])
        norm = (data - vmin) / (vmax - vmin + 1e-10)
        return np.clip(norm, 0, 1)
    
    elif method == "zscore":
        mean, std = np.mean(data), np.std(data)
        norm = (data - mean) / (std + 1e-10)
        norm = np.clip(norm, -3, 3)
        return (norm + 3) / 6
    
    elif method == "percentile":
        p2, p98 = np.percentile(data, [2, 98])
        norm = (data - p2) / (p98 - p2 + 1e-10)
        return np.clip(norm, 0, 1)

def augment_image(data):
    augmented = []
    for angle in [0, 90, 180, 270]:
        rot = ndimage.rotate(data, angle, reshape=False, order=1)
        augmented.append((f'rot{angle}', rot))
        augmented.append((f'rot{angle}_flip', np.fliplr(rot)))
    return augmented

# Test all normalization methods
methods = ["minmax", "zscore", "percentile"]

print("Creating comparison visualizations for every 10th file...")
fits_files = sorted(glob.glob(os.path.join(input_folder, "*.fits")))

# Process every 10th file
comparison_samples = []
for i, fpath in enumerate(fits_files, 1):
    if i % 10 == 0:
        fname = os.path.basename(fpath)
        with fits.open(fpath) as hdul:
            data = hdul[0].data.copy()
            comparison_samples.append((i, fname, data))

print(f"Found {len(comparison_samples)} files to compare (every 10th file)")

# Create individual comparison images for each sample
for sample_num, fname, data in comparison_samples:
    fig, axes = plt.subplots(1, len(methods)+1, figsize=(4*(len(methods)+1), 4))
    
    # Original
    v1, v2 = np.percentile(data, [1, 99])
    axes[0].imshow(data, cmap='gray', vmin=v1, vmax=v2, origin='lower')
    axes[0].set_title(f'Original\n{fname}')
    axes[0].axis('off')
    
    # Each normalization method
    for col_idx, method in enumerate(methods, 1):
        norm = normalize_data(data, method)
        axes[col_idx].imshow(norm, cmap='gray', origin='lower')
        axes[col_idx].set_title(f'{method}')
        axes[col_idx].axis('off')
    
    plt.tight_layout()
    output_path = os.path.join(normalized_folder, f"comparison_{sample_num:04d}_{os.path.splitext(fname)[0]}.png")
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: comparison_{sample_num:04d}_{os.path.splitext(fname)[0]}.png")

print(f"\nAll comparison images saved to '{normalized_folder}' folder")

# Now ask user which method to use
print("\nAvailable normalization methods:")
for i, method in enumerate(methods, 1):
    print(f"  {i}. {method}")

choice = input("\nEnter method number (1-3) or name: ").strip()

if choice.isdigit() and 1 <= int(choice) <= len(methods):
    normalize_method = methods[int(choice)-1]
elif choice in methods:
    normalize_method = choice
else:
    print(f"Invalid choice. Using default: minmax")
    normalize_method = "minmax"

print(f"\nUsing normalization method: {normalize_method}")

# Normalization with chosen method
print("\nStarting normalization...")
viz_samples = []

for i, fpath in enumerate(fits_files, 1):
    fname = os.path.basename(fpath)
    base = os.path.splitext(fname)[0]
    
    with fits.open(fpath) as hdul:
        data = hdul[0].data
        hdr = hdul[0].header
        
        norm = normalize_data(data, normalize_method)
        hdr['NORMTYPE'] = normalize_method
        
        out_path = os.path.join(normalized_folder, f"norm_{base}.fits")
        fits.PrimaryHDU(norm.astype(np.float32), hdr).writeto(out_path, overwrite=True)
        
        if i % 10 == 0:
            viz_samples.append((i, data, norm, fname))
        
        if i % 10 == 0 or i == len(fits_files):
            print(f"  {i}/{len(fits_files)} done")

# Save individual before/after images for the chosen method
if viz_samples:
    print(f"\nSaving before/after comparisons for {normalize_method} method...")
    for num, orig, norm, fname in viz_samples:
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        
        v1, v2 = np.percentile(orig, [1, 99])
        axes[0].imshow(orig, cmap='gray', vmin=v1, vmax=v2, origin='lower')
        axes[0].set_title(f'Original - {fname}')
        axes[0].axis('off')
        
        axes[1].imshow(norm, cmap='gray', origin='lower')
        axes[1].set_title(f'Normalized ({normalize_method})')
        axes[1].axis('off')
        
        plt.tight_layout()
        output_path = os.path.join(normalized_folder, f"final_{num:04d}_{os.path.splitext(fname)[0]}.png")
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  Saved: final_{num:04d}_{os.path.splitext(fname)[0]}.png")

print(f"\nNormalization complete using '{normalize_method}' method!")
print(f"All comparison images saved in '{normalized_folder}' folder")